[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_39_DPO_ORPO_Preference_Tuning.ipynb)

# Lesson 39 — DPO & ORPO: Preference Tuning Without a Reward Model
## Track 3 · Self-hosted & Fine-tuning · Lesson 3 of 5

**Prerequisites:** L38 (QLoRA fine-tuning), L37 (vLLM serving), L36 (capstone). All prior track knowledge assumed.

**What you'll build:** A preference-tuning pipeline that takes a QLoRA SFT model and further aligns it using DPO and ORPO—without a reward model or PPO—then measures the quality delta with an LLM-as-judge.

**Runnable on:** Google Colab T4 (free tier). All heavy computation is optional; every section has a `DRY_RUN` path.

---

## Roadmap

| # | Section | Key Concept |
|---|---------|-------------|
| 1 | Why preference tuning? | SFT imitates data; it never *prefers* good over bad |
| 2 | DPO math | Bradley-Terry + log-ratio loss, no RM needed |
| 3 | ORPO math | SFT + odds-ratio in one pass, no reference model |
| 4 | Preference dataset | (prompt, chosen, rejected) triplets from SQL task |
| 5 | DPO training | `trl.DPOTrainer` on Qwen2.5-1.5B |
| 6 | ORPO training | `trl.ORPOTrainer` — simpler, one stage |
| 7 | Evaluation | SFT vs SFT+DPO vs SFT+ORPO with LLM judge |
| 8 | Pitfalls | 10-row table |
| 9 | Decision tree | When to use SFT / DPO / ORPO / RLHF |
| 10 | Homework | 5 tasks |


## Setup

In [ ]:
# Install — run once
!pip install -q \
    trl>=0.9.0 \
    peft>=0.11.0 \
    bitsandbytes>=0.43.0 \
    transformers>=4.43.0 \
    datasets>=2.20.0 \
    accelerate>=0.31.0 \
    anthropic \
    pandas matplotlib

print("✅ Installation complete")

In [ ]:
import os
import json
import random
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import Dataset
from trl import DPOTrainer, DPOConfig, ORPOTrainer, ORPOConfig, SFTTrainer, SFTConfig

from anthropic import Anthropic

# ── Constants ──────────────────────────────────────────────────────────────────
MODEL_ID      = "Qwen/Qwen2.5-1.5B-Instruct"
MERGED_DIR    = "/content/sql_sft_merged"   # from L38 — or use base model
DPO_OUT_DIR   = "/content/sql_dpo"
ORPO_OUT_DIR  = "/content/sql_orpo"

# Set DRY_RUN=True to skip actual GPU training and observe only code + output shapes
DRY_RUN = not torch.cuda.is_available()

HAIKU   = "claude-haiku-4-5"
SONNET  = "claude-sonnet-4-6"

# Anthropic client (set ANTHROPIC_API_KEY in Colab Secrets)
anthropic_client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", ""))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} | DRY_RUN: {DRY_RUN}")

---
## §1 — Why Preference Tuning?

### The SFT ceiling

L38 taught you SFT (Supervised Fine-Tuning). You fed the model `(prompt, ideal_completion)` pairs. The model learned to *imitate* those completions by minimising cross-entropy.

**The problem:** SFT treats every demo equally — a mediocre demo and an excellent demo get the same gradient signal. The model learns *the average*, not *the best*.

### The RLHF solution (and its cost)

OpenAI's InstructGPT (2022) introduced Reinforcement Learning from Human Feedback (RLHF):

```
1. SFT baseline
2. Collect human preferences: (prompt, chosen_response, rejected_response)
3. Train a reward model (RM) to predict human preference
4. Use PPO (proximal policy optimisation) to fine-tune SFT model → maximise RM score
```

It worked spectacularly — but it's expensive:
- Step 3 needs a **separate RM** (same size as base model)
- Step 4 needs **PPO** — 3× the compute, unstable training, many hyperparams
- Total GPU memory: ~4× the base model size

### DPO: Skip the reward model (Rafailov et al. 2023)

The key insight: **the optimal policy IS a reward model in disguise.** You can rearrange the RLHF optimisation objective algebraically to get a loss that:
- Takes `(prompt, chosen, rejected)` triples directly
- Needs NO separate reward model
- Needs NO PPO
- Needs only one forward pass per triple (+ a frozen reference model)

### ORPO: Skip the reference model too (Hong et al. 2024)

ORPO (Odds Ratio Preference Optimisation) goes further:
- Combines SFT + preference signal in a **single training stage**
- Needs **no reference model** (just the current model)
- Simpler, cheaper, and often matches DPO quality

### Comparison table

| Method | Stages | Extra model | Compute vs SFT | When to use |
|--------|--------|-------------|-----------------|-------------|
| SFT | 1 | — | 1× | Format/task specialisation |
| RLHF | 3 | RM + PPO | 4× | Maximum alignment, big budget |
| **DPO** | **2** | **Frozen ref** | **~2×** | **Preference data available, post-SFT** |
| **ORPO** | **1** | **None** | **~1.2×** | **Preference data, simplicity priority** |

---
## §2 — DPO Math (the intuition you need)

You don't need to re-derive this from scratch, but understanding the loss equation will make every hyperparameter decision obvious.

### Bradley-Terry preference model

Given two responses `y_w` (winner/chosen) and `y_l` (loser/rejected) for prompt `x`, the probability a human prefers `y_w` is:

```
P(y_w > y_l | x) = σ( r*(x, y_w) - r*(x, y_l) )
```

where `r*` is the true human reward and `σ` is the sigmoid.

### The DPO loss

Rafailov et al. showed that under KL-constrained optimisation, the optimal policy satisfies:

```
r*(x, y)  ∝  β · log[ π_θ(y|x) / π_ref(y|x) ]
```

Substituting into Bradley-Terry gives the **DPO loss**:

```
L_DPO = -E[ log σ(
    β · log( π_θ(y_w|x) / π_ref(y_w|x) )   ← reward for chosen
  - β · log( π_θ(y_l|x) / π_ref(y_l|x) )   ← reward for rejected
) ]
```

**In plain English:** minimising this loss pushes the policy to assign
- *relatively higher* probability to `y_w` compared to the reference model
- *relatively lower* probability to `y_l` compared to the reference model

### The β hyperparameter

β controls **how far you're allowed to drift from the reference model**:

| β | Effect |
|---|--------|
| β = 0.01 | Very permissive — model can drift far from ref |
| **β = 0.1** | **Typical starting point** |
| β = 0.5 | Conservative — stay close to reference |
| β → ∞ | Collapses to reference model (no learning) |

### What is the reference model?

In practice: a **frozen copy** of your SFT model loaded at 4-bit (no gradients). Its job is to supply the denominator `π_ref(y|x)` — the baseline log-prob. The gradient only flows through `π_θ`.

### Implementation reality

`trl.DPOTrainer` handles all of this for you. The loss above is computed batch-by-batch. You just supply:
1. `model` — trainable (LoRA adapters on 4-bit base)
2. `ref_model` — frozen (same base, no LoRA, loaded at 4-bit)
3. Dataset with columns `prompt`, `chosen`, `rejected`

---
## §3 — ORPO Math

ORPO's key innovation: you don't need a *separate* reference model if you use the **odds ratio** of the current model itself.

### Odds ratio intuition

For a token sequence `y` and current model `π_θ`:

```
odds(y|x) = P(y|x) / (1 - P(y|x))  =  π_θ(y|x) / (1 - π_θ(y|x))
```

The **log odds ratio** between chosen and rejected:

```
log OR = log odds(y_w|x) - log odds(y_l|x)
       = log[ π_θ(y_w|x)/(1-π_θ(y_w|x)) ] - log[ π_θ(y_l|x)/(1-π_θ(y_l|x)) ]
```

This quantity is *large and positive* when the model strongly prefers the winner. We penalise when it's not.

### The ORPO loss

```
L_ORPO = L_SFT  +  λ · L_OR

where:
  L_SFT = cross-entropy on chosen sequences (standard language modelling)
  L_OR  = -log σ( log OR )   ← encourages high log OR
  λ     = weight (default 0.1)
```

**Why does this work without a reference model?**  
The odds ratio is *contrastive within the current model* — it compares how the model treats `y_w` vs `y_l` right now. No frozen snapshot needed.

### DPO vs ORPO

| Dimension | DPO | ORPO |
|-----------|-----|------|
| Reference model | Required (frozen SFT) | **Not needed** |
| Training stages | 2 (SFT → DPO) | **1 (SFT + pref together)** |
| VRAM overhead | ~2× base | ~1.2× base |
| Key hyperparameter | β (KL penalty) | λ (OR weight) |
| Best when | You have a strong SFT checkpoint | You're training from scratch |
| Typical quality | Slightly higher ceiling | Slightly lower ceiling |
| Code complexity | Medium | Low |

---
## §4 — Preference Dataset

DPO/ORPO both need `(prompt, chosen, rejected)` triples. We'll build a small synthetic dataset continuing the SQL generation task from L38.

### How real preference data is collected

1. **Human labellers** rank model outputs → expensive but ground-truth
2. **AI feedback (RLAIF)** — a strong model judges outputs → scalable, Constitutional AI
3. **Heuristic rules** — correct SQL vs wrong SQL, test passes vs fails → programmatic
4. **Existing datasets** — `argilla/dpo-mix-7k`, `HuggingFaceH4/ultrafeedback_binarized`

For teaching, we'll use approach 3: generate two SQL outputs per prompt, verify correctness with a simple heuristic, assign chosen/rejected.

In production with L38's SQL task, you'd run both the base model and the SFT model on every test prompt, then use an LLM judge (or SQL execution) to label them.

In [ ]:
# ── §4 Preference Dataset Builder ──────────────────────────────────────────────
# We create synthetic (prompt, chosen, rejected) triples.
# In a real setting: run two model variants, have a judge label them.

SQL_EXAMPLES = [
    {
        "schema": "CREATE TABLE employees (id INT, name TEXT, dept TEXT, salary FLOAT, hire_date DATE);",
        "question": "Find the top 3 highest-paid employees in the Engineering department.",
        "chosen": "SELECT id, name, salary FROM employees WHERE dept = 'Engineering' ORDER BY salary DESC LIMIT 3;",
        "rejected": "SELECT * FROM employees WHERE dept = Engineering ORDER BY salary LIMIT 3;",
        # rejected issues: missing quotes around string, wrong sort direction, SELECT *
    },
    {
        "schema": "CREATE TABLE orders (order_id INT, customer_id INT, total FLOAT, status TEXT, created_at TIMESTAMP);",
        "question": "Count the number of completed orders per customer, only showing customers with more than 5.",
        "chosen": "SELECT customer_id, COUNT(*) AS order_count FROM orders WHERE status = 'completed' GROUP BY customer_id HAVING COUNT(*) > 5;",
        "rejected": "SELECT customer_id, COUNT(*) FROM orders WHERE status = completed GROUP BY customer_id WHERE COUNT(*) > 5;",
        # rejected issues: HAVING replaced with WHERE (invalid), missing quotes
    },
    {
        "schema": "CREATE TABLE products (product_id INT, name TEXT, price FLOAT, category TEXT, stock INT);",
        "question": "Find all products in the Electronics category that are out of stock.",
        "chosen": "SELECT product_id, name, price FROM products WHERE category = 'Electronics' AND stock = 0;",
        "rejected": "SELECT * FROM products WHERE category = Electronics AND stock < 0;",
        # rejected issues: missing quotes, stock < 0 is wrong (out of stock = 0, not negative)
    },
    {
        "schema": "CREATE TABLE students (id INT, name TEXT, grade FLOAT, course TEXT, teacher TEXT);",
        "question": "What is the average grade per course, ordered from highest to lowest?",
        "chosen": "SELECT course, AVG(grade) AS avg_grade FROM students GROUP BY course ORDER BY avg_grade DESC;",
        "rejected": "SELECT course, AVG(grade) FROM students GROUP BY course ORDER BY AVG(grade) ASC;",
        # rejected: uses ASC instead of DESC, doesn't alias for clarity
    },
    {
        "schema": "CREATE TABLE logs (log_id INT, user_id INT, action TEXT, ts TIMESTAMP, ip TEXT);",
        "question": "Find users who performed more than 10 login actions in the last 7 days.",
        "chosen": "SELECT user_id, COUNT(*) AS login_count FROM logs WHERE action = 'login' AND ts >= NOW() - INTERVAL '7 days' GROUP BY user_id HAVING COUNT(*) > 10;",
        "rejected": "SELECT user_id FROM logs WHERE action = login GROUP BY user_id HAVING COUNT(*) > 10;",
        # rejected: missing quotes, missing date filter
    },
    {
        "schema": "CREATE TABLE invoices (id INT, vendor TEXT, amount FLOAT, paid BOOLEAN, due_date DATE);",
        "question": "List all unpaid invoices ordered by due date ascending, showing vendor and amount.",
        "chosen": "SELECT vendor, amount, due_date FROM invoices WHERE paid = FALSE ORDER BY due_date ASC;",
        "rejected": "SELECT * FROM invoices WHERE paid = 0 ORDER BY amount;",
        # rejected: SELECT *, wrong column in ORDER BY, boolean comparison via int is db-dependent
    },
    {
        "schema": "CREATE TABLE reviews (id INT, product_id INT, rating INT, comment TEXT, created DATE);",
        "question": "Get products with an average rating above 4.5 that have at least 10 reviews.",
        "chosen": "SELECT product_id, AVG(rating) AS avg_rating, COUNT(*) AS review_count FROM reviews GROUP BY product_id HAVING AVG(rating) > 4.5 AND COUNT(*) >= 10;",
        "rejected": "SELECT product_id, AVG(rating) FROM reviews WHERE rating > 4.5 GROUP BY product_id HAVING COUNT(*) >= 10;",
        # rejected: filters individual ratings > 4.5 instead of avg, misses avg in SELECT
    },
    {
        "schema": "CREATE TABLE flights (flight_id INT, origin TEXT, destination TEXT, price FLOAT, seats_left INT);",
        "question": "Find the cheapest available flight from NYC to LAX.",
        "chosen": "SELECT flight_id, price, seats_left FROM flights WHERE origin = 'NYC' AND destination = 'LAX' AND seats_left > 0 ORDER BY price ASC LIMIT 1;",
        "rejected": "SELECT * FROM flights WHERE origin = NYC AND destination = LAX ORDER BY price LIMIT 1;",
        # rejected: missing quotes, doesn't check availability (seats_left > 0)
    },
]

SYSTEM_PROMPT = "You are a SQL expert. Write precise, correct SQL queries based on the provided schema and question."

def make_preference_row(example: dict, tokenizer=None) -> dict:
    """Format one SQL example into (prompt, chosen, rejected) with chat template."""
    user_msg = f"Schema:\n{example['schema']}\n\nQuestion: {example['question']}"
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    
    # DPO format: prompt is the conversation up to the assistant turn
    # chosen/rejected are the assistant responses (plain text)
    return {
        "prompt": messages,          # list of dicts, trl handles chat template
        "chosen":  [{"role": "assistant", "content": example["chosen"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}],
    }

raw_rows = [make_preference_row(ex) for ex in SQL_EXAMPLES]

# Show one example
print("=== Example preference row ===")
print(f"Prompt messages: {len(raw_rows[0]['prompt'])} turns")
print(f"Chosen:  {raw_rows[0]['chosen'][0]['content'][:80]}...")
print(f"Rejected: {raw_rows[0]['rejected'][0]['content'][:80]}...")
print(f"\nTotal preference pairs: {len(raw_rows)}")

In [ ]:
# Split into train/eval (small dataset for demo — production use 500+ pairs)
random.seed(42)
random.shuffle(raw_rows)

n_train = 6
train_rows = raw_rows[:n_train]
eval_rows  = raw_rows[n_train:]

train_dataset = Dataset.from_list(train_rows)
eval_dataset  = Dataset.from_list(eval_rows)

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print(f"\nDataset features: {train_dataset.features}")

---
## §5 — DPO Training

### Memory budget

DPO needs **both** the trainable model and the frozen reference in memory:

| Component | Size (Qwen2.5-1.5B INT4) | Notes |
|-----------|--------------------------|-------|
| Base model (trainable) | ~1.2 GB | LoRA adapters on top |
| Reference model (frozen) | ~1.2 GB | No adapters, just base |
| LoRA adapters | ~50 MB | r=16, target all *_proj |
| Optimizer states | ~200 MB | paged_adamw_32bit |
| Activations + KV cache | ~1 GB | Gradient checkpointing helps |
| **Total** | **~3.7 GB** | **Fits T4 (15GB) comfortably** |

### Key hyperparameters for DPO

| Parameter | Value | Why |
|-----------|-------|-----|
| `beta` | 0.1 | KL penalty — standard starting point |
| `learning_rate` | 5e-5 | Lower than SFT (smaller gradient signal) |
| `num_train_epochs` | 1–3 | More epochs risks overfit on small pref datasets |
| `max_length` | 512 | Total prompt+chosen+rejected token budget |
| `max_prompt_length` | 256 | Prompt portion of the above |
| `loss_type` | "sigmoid" | Classic DPO; alternatives: "hinge", "ipo" |

In [ ]:
# ── §5 DPO Setup ───────────────────────────────────────────────────────────────

def load_model_for_dpo(model_id: str, is_reference: bool = False):
    """Load model with 4-bit quantisation. Reference model gets no LoRA."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    if is_reference:
        # Freeze everything — reference model provides π_ref
        for p in model.parameters():
            p.requires_grad = False
        model.eval()
        print(f"Reference model loaded ({model_id}) — fully frozen")
    else:
        # Add LoRA for trainable model
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                             "gate_proj", "up_proj", "down_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
        print(f"Trainable model loaded ({model_id}) — LoRA adapters active")
    return model


def run_dpo_training():
    """Full DPO training run — skipped in DRY_RUN."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"   # DPO requires left-padding

    # Load trainable model (from L38 merged checkpoint if available, else base)
    source = MERGED_DIR if os.path.exists(MERGED_DIR) else MODEL_ID
    print(f"Loading trainable model from: {source}")
    model     = load_model_for_dpo(source, is_reference=False)
    ref_model = load_model_for_dpo(MODEL_ID, is_reference=True)  # always base for ref

    dpo_config = DPOConfig(
        output_dir=DPO_OUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        bf16=True,
        beta=0.1,                      # KL penalty
        max_length=512,
        max_prompt_length=256,
        loss_type="sigmoid",           # classic DPO
        optim="paged_adamw_32bit",
        save_strategy="epoch",
        logging_steps=1,
        remove_unused_columns=False,
    )

    trainer = DPOTrainer(
        model=model,
        ref_model=ref_model,
        args=dpo_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    print("\n🚀 Starting DPO training...")
    trainer.train()
    trainer.save_model(DPO_OUT_DIR)
    tokenizer.save_pretrained(DPO_OUT_DIR)
    print(f"✅ DPO model saved to {DPO_OUT_DIR}")
    return trainer


if DRY_RUN:
    print("DRY_RUN mode — printing DPO config only (no GPU detected)")
    print(textwrap.dedent("""
    DPOConfig(
        beta=0.1,               # KL penalty — lower = more deviation from ref
        loss_type='sigmoid',    # Classic DPO loss
        max_length=512,         # prompt + chosen/rejected total tokens
        max_prompt_length=256,  # tokens allocated to prompt
        learning_rate=5e-5,     # lower than SFT (1e-4) — shallower gradient
        num_train_epochs=1,
        padding_side='left',    # REQUIRED for DPO — model sees response right-aligned
    )
    """))
else:
    dpo_trainer = run_dpo_training()

In [ ]:
# ── DPO Training Metrics Visualisation ─────────────────────────────────────────
# When DRY_RUN=False, uncomment the trainer.state.log_history block.
# Here we simulate a plausible DPO loss curve for illustration.

import numpy as np

# Simulated DPO training metrics (replace with trainer.state.log_history in real run)
sim_steps  = list(range(1, 7))
sim_loss   = [0.693, 0.612, 0.571, 0.544, 0.523, 0.508]  # decreasing DPO loss
sim_reward_acc = [0.50, 0.57, 0.62, 0.66, 0.70, 0.73]     # fraction chosen > rejected
sim_margin = [0.00, 0.08, 0.14, 0.19, 0.24, 0.28]         # log-prob margin

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(sim_steps, sim_loss, 'b-o')
axes[0].set_title('DPO Loss (lower = better)'); axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].axhline(y=0.693, color='gray', linestyle='--', label='Random (log 2)', alpha=0.5)
axes[0].legend()

axes[1].plot(sim_steps, sim_reward_acc, 'g-o')
axes[1].set_title('Reward Accuracy (fraction chosen > rejected)')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Accuracy')
axes[1].axhline(y=0.5, color='gray', linestyle='--', label='Random baseline', alpha=0.5)
axes[1].legend()

axes[2].plot(sim_steps, sim_margin, 'r-o')
axes[2].set_title('Reward Margin (log-prob gap chosen−rejected)')
axes[2].set_xlabel('Step'); axes[2].set_ylabel('Margin')

plt.suptitle('DPO Training Metrics (simulated)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/dpo_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

print("""
Reading DPO metrics:
  DPO loss starts near log(2) ≈ 0.693 (random — chosen prob = rejected prob)
  It should decrease as the model learns to prefer chosen responses.
  Reward accuracy > 0.5 means model assigns higher reward to chosen than rejected.
  Reward margin increasing = the gap is widening — good signal.
  
  ⚠ Watch for: reward accuracy stalling at 0.5 → β too high, LR too low, or bad data.
""")

---
## §6 — ORPO Training

ORPO is cleaner: load one model, one trainer, done.

### Key differences from DPO setup
- No `ref_model` argument
- Use `ORPOConfig` with `lambda_` (OR weight, called `lambda` in the paper)
- Training typically from a *pretrained* or *lightly SFT* checkpoint (not a fully-fine-tuned one)
- The SFT signal inside ORPO means it still teaches format, so you don't need a prior SFT stage

In [ ]:
# ── §6 ORPO Training ───────────────────────────────────────────────────────────

def run_orpo_training():
    """Full ORPO training run — skipped in DRY_RUN."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    # ORPO: load base model (ORPO does SFT+pref together, so start from base)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    orpo_config = ORPOConfig(
        output_dir=ORPO_OUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=8e-5,            # ORPO uses slightly higher LR than DPO
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        bf16=True,
        lambda_=0.1,                    # ORPO: weight of odds-ratio loss vs SFT
        max_length=512,
        max_prompt_length=256,
        optim="paged_adamw_32bit",
        save_strategy="epoch",
        logging_steps=1,
        remove_unused_columns=False,
    )

    # ORPO trainer — NOTE: no ref_model argument!
    trainer = ORPOTrainer(
        model=model,
        args=orpo_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    print("\n🚀 Starting ORPO training...")
    trainer.train()
    trainer.save_model(ORPO_OUT_DIR)
    tokenizer.save_pretrained(ORPO_OUT_DIR)
    print(f"✅ ORPO model saved to {ORPO_OUT_DIR}")
    return trainer


if DRY_RUN:
    print("DRY_RUN mode — printing ORPO config diff vs DPO")
    print(textwrap.dedent("""
    ORPO vs DPO config differences:
    
    DPO:                          ORPO:
    ─────────────────────────     ─────────────────────────
    ref_model=frozen_base         (no ref_model!)
    DPOConfig(beta=0.1)           ORPOConfig(lambda_=0.1)
    loss = DPO objective          loss = L_SFT + λ·L_OR
    start from SFT checkpoint     start from base or light SFT
    learning_rate = 5e-5          learning_rate = 8e-5
    """))
else:
    orpo_trainer = run_orpo_training()

---
## §7 — Evaluation: SFT vs SFT+DPO vs SFT+ORPO

We compare the three model variants on held-out prompts using:
1. **Heuristic correctness** — basic SQL syntax checks
2. **LLM-as-judge** — Haiku rates each response on a rubric

In [ ]:
# ── §7 Evaluation ──────────────────────────────────────────────────────────────

TEST_CASES = [
    {
        "schema": "CREATE TABLE sales (id INT, rep TEXT, region TEXT, amount FLOAT, quarter INT);",
        "question": "Find the total sales amount per region for Q1, sorted highest first.",
        "reference": "SELECT region, SUM(amount) AS total FROM sales WHERE quarter = 1 GROUP BY region ORDER BY total DESC;",
    },
    {
        "schema": "CREATE TABLE users (id INT, email TEXT, plan TEXT, active BOOLEAN, joined DATE);",
        "question": "Count active users on each plan type.",
        "reference": "SELECT plan, COUNT(*) AS user_count FROM users WHERE active = TRUE GROUP BY plan;",
    },
    {
        "schema": "CREATE TABLE tickets (id INT, priority TEXT, status TEXT, assigned_to TEXT, created_at TIMESTAMP);",
        "question": "List all open high-priority tickets assigned to 'alice', newest first.",
        "reference": "SELECT id, priority, created_at FROM tickets WHERE status = 'open' AND priority = 'high' AND assigned_to = 'alice' ORDER BY created_at DESC;",
    },
]


def heuristic_sql_score(sql: str) -> dict:
    """Quick structural checks on generated SQL."""
    sql_upper = sql.upper().strip()
    return {
        "starts_with_select": sql_upper.startswith("SELECT"),
        "has_from":            "FROM" in sql_upper,
        "quoted_strings":      "'" in sql or '"' not in sql,  # no unquoted identifiers
        "no_select_star":      "SELECT *" not in sql_upper,
        "has_semicolon":       sql.strip().endswith(";"),
    }


def llm_judge_sql(schema: str, question: str, sql: str, reference: str) -> dict:
    """Haiku judges the SQL on a 0-10 scale with sub-scores."""
    prompt = f"""You are a SQL quality judge.

Schema:
{schema}

Question: {question}

Reference (correct) SQL:
{reference}

Generated SQL:
{sql}

Score the generated SQL on:
- correctness (0-4): Does it correctly answer the question?
- syntax (0-2): Is it valid SQL syntax?
- schema_adherence (0-2): Does it only use columns/tables from the schema?
- style (0-2): Is it clean, aliased, readable?

Return JSON: {{"correctness": int, "syntax": int, "schema_adherence": int, "style": int, "total": int, "verdict": "correct|partial|incorrect"}}"""

    try:
        resp = anthropic_client.messages.create(
            model=HAIKU,
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}],
        )
        text = resp.content[0].text.strip()
        # Strip code fences if present
        if text.startswith("```"):
            text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return json.loads(text)
    except Exception as e:
        return {"correctness": 0, "syntax": 0, "schema_adherence": 0, "style": 0,
                "total": 0, "verdict": "error", "error": str(e)}


def generate_sql(model, tokenizer, schema: str, question: str,
                 max_new_tokens: int = 150) -> str:
    """Generate SQL from a loaded model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Schema:\n{schema}\n\nQuestion: {question}"},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print("Evaluation utilities loaded.")
print("These functions will be called after model loading in the next cell.")

In [ ]:
# ── Run evaluation comparing model variants ─────────────────────────────────────
# In DRY_RUN: use pre-baked outputs that simulate what models would generate

SIMULATED_OUTPUTS = {
    "base": [
        # Base model outputs — often missing WHERE filters, wrong sort direction
        "SELECT region, SUM(amount) FROM sales GROUP BY region ORDER BY SUM(amount);",
        "SELECT plan, COUNT(*) FROM users GROUP BY plan;",
        "SELECT * FROM tickets WHERE priority = high AND assigned_to = alice ORDER BY created_at;",
    ],
    "sft": [
        # SFT model — better format, but occasional logic errors
        "SELECT region, SUM(amount) AS total FROM sales WHERE quarter = 1 GROUP BY region ORDER BY total DESC;",
        "SELECT plan, COUNT(*) AS user_count FROM users WHERE active = TRUE GROUP BY plan;",
        "SELECT id, priority, created_at FROM tickets WHERE status = 'open' AND priority = 'high' AND assigned_to = alice ORDER BY created_at DESC;",
    ],
    "dpo": [
        # DPO model — correct SQL, proper quoting, aliases
        "SELECT region, SUM(amount) AS total FROM sales WHERE quarter = 1 GROUP BY region ORDER BY total DESC;",
        "SELECT plan, COUNT(*) AS user_count FROM users WHERE active = TRUE GROUP BY plan;",
        "SELECT id, priority, created_at FROM tickets WHERE status = 'open' AND priority = 'high' AND assigned_to = 'alice' ORDER BY created_at DESC;",
    ],
    "orpo": [
        # ORPO model — similar to DPO, one minor style difference
        "SELECT region, SUM(amount) AS total_sales FROM sales WHERE quarter = 1 GROUP BY region ORDER BY total_sales DESC;",
        "SELECT plan, COUNT(*) AS active_users FROM users WHERE active = TRUE GROUP BY plan ORDER BY active_users DESC;",
        "SELECT id, priority, created_at FROM tickets WHERE status = 'open' AND priority = 'high' AND assigned_to = 'alice' ORDER BY created_at DESC;",
    ],
}

results = []

if not DRY_RUN:
    print("Loading models for evaluation...")
    # Load each model variant
    # ... (same load_model_for_dpo pattern + merge_and_unload for DPO/ORPO)
    print("(Set DRY_RUN=False and uncomment to run live evaluation)")

# Use simulated outputs (both DRY_RUN and demo)
print("Running LLM-judge evaluation on simulated outputs...\n")

for variant, outputs in SIMULATED_OUTPUTS.items():
    for i, (test, sql) in enumerate(zip(TEST_CASES, outputs)):
        heur = heuristic_sql_score(sql)
        judge = llm_judge_sql(
            test["schema"], test["question"], sql, test["reference"]
        )
        heuristic_pass = sum(heur.values()) / len(heur)
        results.append({
            "variant":        variant,
            "test_case":      i,
            "sql":            sql[:60] + "...",
            "heuristic_pass": heuristic_pass,
            "llm_total":      judge.get("total", 0),
            "verdict":        judge.get("verdict", "error"),
            "correctness":    judge.get("correctness", 0),
            "syntax":         judge.get("syntax", 0),
        })
        print(f"  [{variant}] case {i}: verdict={judge.get('verdict','?')} total={judge.get('total',0)}/10")

df = pd.DataFrame(results)
print("\n=== Results per variant ===")
summary = df.groupby("variant")[["heuristic_pass", "llm_total", "correctness", "syntax"]].mean().round(2)
print(summary.to_string())

In [ ]:
# ── Visualise quality delta ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

variant_order = ["base", "sft", "dpo", "orpo"]
colors = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db"]

# LLM judge total score
means = [df[df.variant == v]["llm_total"].mean() for v in variant_order]
bars = axes[0].bar(variant_order, means, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title("LLM Judge Score (0–10)", fontsize=13)
axes[0].set_ylabel("Average score")
axes[0].set_ylim(0, 10)
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                 f"{val:.1f}", ha='center', fontsize=11, fontweight='bold')

# Heuristic pass rate
hmeans = [df[df.variant == v]["heuristic_pass"].mean() for v in variant_order]
bars2 = axes[1].bar(variant_order, hmeans, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title("Heuristic Pass Rate", fontsize=13)
axes[1].set_ylabel("Pass rate")
axes[1].set_ylim(0, 1)
for bar, val in zip(bars2, hmeans):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f"{val:.2f}", ha='center', fontsize=11, fontweight='bold')

plt.suptitle("Quality Delta: Base → SFT → DPO / ORPO", fontsize=14)
plt.tight_layout()
plt.savefig("/content/preference_tuning_eval.png", dpi=100, bbox_inches='tight')
plt.show()

# Delta table
base_score  = df[df.variant == "base"]["llm_total"].mean()
sft_score   = df[df.variant == "sft"]["llm_total"].mean()
dpo_score   = df[df.variant == "dpo"]["llm_total"].mean()
orpo_score  = df[df.variant == "orpo"]["llm_total"].mean()

print("\n=== Quality Delta ===")
print(f"  Base  → SFT :  +{sft_score - base_score:.1f} pts  ({(sft_score/base_score - 1)*100:.0f}% gain)")
print(f"  SFT   → DPO :  +{dpo_score - sft_score:.1f} pts  ({(dpo_score/sft_score - 1)*100:.0f}% gain)")
print(f"  SFT   → ORPO:  +{orpo_score - sft_score:.1f} pts  ({(orpo_score/sft_score - 1)*100:.0f}% gain)")
print(f"  DPO vs ORPO :  {dpo_score - orpo_score:+.1f} pts")

---
## §8 — Pitfalls Table

| # | Pitfall | Symptom | Fix |
|---|---------|---------|-----|
| 1 | **Reward accuracy stuck at 0.5** | DPO loss barely moves | Check β — too high means model can't deviate; also check data quality (chosen vs rejected too similar) |
| 2 | **Reward hacking** | Judge score improves but real quality drops | Preference labels must reflect real quality, not surface style; use diverse eval |
| 3 | **Right-padding instead of left** | `IndexError` or garbage outputs | DPO/ORPO *require* `padding_side='left'`; the response must be right-aligned |
| 4 | **Reference model on wrong device** | OOM or shape mismatch | Ensure both trainable model and ref model use `device_map='auto'` |
| 5 | **Preference data from the same model** | Circular: model learns to prefer its own outputs | Chosen/rejected should come from different models or quality signals, not the same model |
| 6 | **β too low** | Model diverges from reference, starts hallucinating | Start at β=0.1, increase if model diverges |
| 7 | **ORPO λ too high** | Odds-ratio signal overwhelms SFT; format degrades | Start at λ=0.1; tune carefully |
| 8 | **Using DPO on a very early SFT model** | Chosen/rejected distinction meaningless when SFT is bad | Ensure SFT is reasonably trained before DPO; DPO needs a competent reference |
| 9 | **Tiny preference dataset (< 100 pairs)** | Overfit — high reward accuracy but poor generalisation | Augment with public pref datasets (UltraFeedback, DPO-Mix) or generate more pairs |
| 10 | **No eval during training** | Loss decreases but reward accuracy plateaus and you miss it | Always set `eval_dataset` and check `rewards/accuracies` in logs |

---
## §9 — Decision Tree: Which Method to Use?

```
Do you have preference data (chosen/rejected pairs)?
   │
   ├── NO → Use SFT (L38)
   │
   └── YES
         │
         ├── Do you already have a strong SFT checkpoint?
         │       ├── YES → Use DPO (post-SFT preference tuning)
         │       └── NO  → Use ORPO (SFT + pref in one stage)
         │
         ├── Budget and scale for training a reward model?
         │       └── YES + large team + production LLM → Consider RLHF / PPO
         │
         └── Is the task highly constrained (SQL correctness, code tests)?
                 └── Use heuristic rejection sampling (generate N, keep only passing ones)
                     then SFT on the filtered set (simpler than DPO, often just as good)
```

### Practical guideline

| Scenario | Recommendation |
|----------|---------------|
| Starting from scratch, have pref data | ORPO |
| Have strong SFT, want quality uplift | DPO (β=0.1) |
| Have executable test suite (SQL, code) | Rejection sampling → SFT |
| Large team, multiple preference raters | RLHF/PPO |
| Only format improvement needed | SFT is enough |


---
## §10 — Homework

1. **β sweep**: Run DPO with β ∈ {0.01, 0.05, 0.1, 0.5} on the same dataset. Plot reward accuracy vs β. At what β does the model stop learning?

2. **ORPO λ sweep**: Same experiment for ORPO with λ ∈ {0.01, 0.05, 0.1, 0.5}. What happens to SFT loss vs OR loss as λ increases?

3. **Preference data from execution**: Use Python's `sqlite3` to actually execute the chosen and rejected SQLs against an in-memory database. Replace the heuristic score with a binary pass/fail from execution. Does the judge score correlate with execution success?

4. **Wire DPO model into AutoResearcher**: Take the DPO-fine-tuned model, serve it with vLLM (L37), and add it as a new tier in the L31 reliability spine: `self_hosted_dpo → haiku → static`. Measure whether the DPO model improves the self-hosted tier's answer quality.

5. **Rejection sampling baseline**: Generate 5 SQL candidates per prompt using the SFT model with temperature=0.8, run the heuristic scorer on each, keep only those with >0.8 pass rate, and fine-tune a fresh model on those filtered examples. Compare quality vs DPO. This is often the most practical approach for deterministic tasks.

---
## Track 3 Progress & L40 Preview

```
Track 3: Self-hosted & Fine-tuning
  L37 ✅ vLLM — serve your own model
  L38 ✅ QLoRA — fine-tune on real data
  L39 ✅ DPO/ORPO — preference tuning without RM
  L40 ⏳ Knowledge Distillation — teach a small model with a big teacher
  L41    Model Merging — SLERP, TIES, DARE
```

**L40 preview — Knowledge Distillation:**

You have a 1.5B model (fast, cheap) and a 70B model (slow, expensive). Can you make the 1.5B model behave like the 70B on your specific task?

Knowledge distillation says yes:
- The 70B **teacher** generates soft probability distributions (not just tokens)
- The 1.5B **student** minimises KL-divergence against those distributions
- Result: student captures the teacher's uncertainty and nuance, not just its best guess

**Coming up:** `SequenceKD` (generate teacher outputs, SFT student on them) → `ImplicitKD` (student matches teacher's token-level log-probs) → distillation vs fine-tuning comparison → how models like Phi-2 and Gemma-2B were built this way.


In [ ]:
# ── Lesson Summary ──────────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  Lesson 39 — DPO & ORPO: Preference Tuning Without a Reward Model  ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  KEY TAKEAWAYS                                                       ║
║  ─────────────────────────────────────────────────────────────────  ║
║  1. SFT imitates demos; DPO/ORPO teach the model to PREFER better   ║
║     responses over worse ones using (prompt, chosen, rejected) data  ║
║                                                                      ║
║  2. DPO: no reward model, no PPO. Uses a frozen SFT copy as the     ║
║     reference. β controls KL divergence from reference.              ║
║                                                                      ║
║  3. ORPO: no reference model at all. Combines SFT + odds-ratio      ║
║     penalty in one stage. λ controls the preference signal weight.   ║
║                                                                      ║
║  4. Reward accuracy is your main training signal — it should rise   ║
║     above 0.5 quickly. If not, check β/λ, LR, and data quality.     ║
║                                                                      ║
║  5. For deterministic tasks (SQL, code), rejection sampling on an   ║
║     SFT model is often simpler and just as effective as DPO.         ║
║                                                                      ║
║  DECISION RULE                                                       ║
║  Have strong SFT + pref data → DPO                                   ║
║  Starting fresh + pref data  → ORPO                                  ║
║  Executable test suite       → Rejection sampling → SFT             ║
║                                                                      ║
║  NEXT: L40 — Knowledge Distillation (big teacher → small student)   ║
╚══════════════════════════════════════════════════════════════════════╝
""")